# Lecture 39: Putting It All Together — Four Ways to Predict a Score
v.ekc

The semester in one problem: a student scored 21/40 on Midterm 1 and missed Midterm 2. What score should we give them? Four increasingly-good answers — then a real observational question about whether tutoring helped. (No matching deck — this one lives in the notebook.)

**Today's sections:**
1. The setup
2. Option 1 — scale up
3. Option 2 — z-score
4. Option 3 — percentile
5. Option 4 — regression
6. Did tutoring help?

In [ ]:
from datascience import *
import numpy as np

import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

---
## 1. The Setup

In [ ]:
scores = Table.read_table("scores.csv")
scores.drop(2).show(5)

In [ ]:
mt1 = scores.column('Midterm 1')
mt2 = scores.column('Midterm 2')
print('Midterm 1 avg:', np.average(mt1), 'std dev:', np.std(mt1))
print('Midterm 2 avg:', np.average(mt2), 'std dev:', np.std(mt2))

---
## 2. Option 1 — Scale Up

21/40 → 26.25/50. Simple… and wrong. Look where it lands in standard units compared to where the student actually stood:

In [ ]:
mt1_actual = 21
mt2_estimate_1 = mt1_actual / 40 * 50
mt2_estimate_1

In [ ]:
mt2_estimate_1 - np.average(mt2)

In [ ]:
(mt2_estimate_1 - np.average(mt2)) / np.std(mt2)

In [ ]:
(mt1_actual - np.average(mt1)) / np.std(mt1)

> **The flaw:** scaling up preserves the *fraction*, not the *standing*. The student was +0.15 SDs on MT1; the scaled score puts them at a different z on MT2 because the exams have different means and spreads.

---
## 3. Option 2 — Z-Score

Better: keep the student's standard units the same on both exams.

In [ ]:
mt1_actual = 21
mt1_z = (mt1_actual - np.average(mt1)) / np.std(mt1)
mt2_estimate_2 = np.average(mt2) + mt1_z * np.std(mt2)
mt2_estimate_2

In [ ]:
mt2_estimate_2 - np.average(mt2)

In [ ]:
(mt2_estimate_2 - np.average(mt2)) / np.std(mt2)

In [ ]:
scores.hist('Midterm 1', unit='point')

In [ ]:
scores.hist('Midterm 2', unit='point')

---
## 4. Option 3 — Percentile

Or keep their *rank*: find their MT1 percentile, hand them the same percentile of MT2.

In [ ]:
mt1_actual = 21
mt1_percentile = sum(mt1 <= mt1_actual) / len(mt1) * 100
mt1_percentile

In [ ]:
percentile(mt1_percentile, mt1)

In [ ]:
percentile(mt1_percentile, mt2)

In [ ]:
scores.where('Midterm 1', 21).hist('Midterm 2', normed=False)

---
## 5. Option 4 — Regression

But scores aren't perfectly correlated — r ≈ 0.6 — so the best prediction **regresses toward the mean**: same z-score idea, shrunk by r.

In [ ]:
def standard_units(arr):
    """Converts an array to standard units """
    return (arr - np.average(arr))/np.std(arr)

def correlation(t, x, y):
    """Computes correlation: t is a table, and x and y are column names """
    x_standard = standard_units(t.column(x))
    y_standard = standard_units(t.column(y))
    return np.average(x_standard * y_standard)

def slope(t, x, y):
    """Computes the slope of the regression line, like correlation above """
    r = correlation(t, x, y)
    y_sd = np.std(t.column(y))
    x_sd = np.std(t.column(x))
    return r * y_sd / x_sd

def intercept(t, x, y):
    """Computes the intercept of the regression line, like slope above """
    x_mean = np.mean(t.column(x))
    y_mean = np.mean(t.column(y))
    return y_mean - slope(t, x, y)*x_mean

def fitted_values(t, x, y):
    """Return an array of the regression estimates (predictions) at all the x values"""
    a = slope(t, x, y)
    b = intercept(t, x, y)
    return a*t.column(x) + b

In [ ]:
r = correlation(scores, 'Midterm 1', 'Midterm 2')
r

In [ ]:
mt1_actual = 21
mt1_z = (mt1_actual - np.average(mt1)) / np.std(mt1)
mt2_estimate_2 = np.average(mt2) + mt1_z * r * np.std(mt2)
mt2_estimate_2

In [ ]:
scores.scatter('Midterm 1', 'Midterm 2')

In [ ]:
a = slope(scores, 'Midterm 1', 'Midterm 2')
b = intercept(scores, 'Midterm 1', 'Midterm 2')
scores.drop(2).with_column('Fitted', a * mt1 + b ).scatter('Midterm 1')

In [ ]:
scores.with_column('Residual', mt2 - (a * mt1 + b)).scatter('Midterm 1', 'Residual')

In [ ]:
scores.where("Midterm 1", mt1_actual).hist('Midterm 2')

In [ ]:
scores.where("Midterm 1", are.between_or_equal_to(mt1_actual-2, mt1_actual+2)).hist('Midterm 2')

In [ ]:
def avg_mt2(mt1):
    near = scores.where("Midterm 1", are.between_or_equal_to(mt1-2, mt1+2))
    return near.column("Midterm 2").mean()

avg_mt2(mt1_actual)

In [ ]:
mt2_avg = scores.apply(avg_mt2, 'Midterm 1')

In [ ]:
scores.drop(2).with_column('Avg', mt2_avg).scatter('Midterm 1')

> **Note what regression did:** prediction = average + r · z · SD. Options 2 and 3 assume perfect correlation; regression admits the exams only partly agree, so it pulls the estimate toward the middle. The graph of averages confirms it.

### Try It 1: A different student 😊

1. Use the fitted `a` (slope) and `b` (intercept) to predict Midterm 2 for a student who scored 30 on Midterm 1. How does it compare with the pure z-score estimate?

In [ ]:
# 1. regression prediction at mt1 = 30


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*One line — and it lands closer to the MT2 average than the z-score method would, because r < 1.*

```python
# 1.
a * 30 + b
```

</details>

---
## 6. Did Tutoring Help? 🎓

Some students were mentored between the exams. Mentored students scored *lower* on MT2 on average — did tutoring hurt?! Careful: **who signs up for tutoring?** Compare each student to the MT2 score *predicted from non-mentored students with the same MT1*:

In [ ]:
scores.show(5)

In [ ]:
scores.scatter('Midterm 1', 'Midterm 2', group='Mentored')

In [ ]:
scores.hist('Midterm 1', group='Mentored', bins=np.arange(0, 41, 5), normed=False)

In [ ]:
scores.hist('Midterm 2', group='Mentored', bins=np.arange(0, 51, 5), normed=False)

In [ ]:
no_mentor = scores.where("Mentored", False)

def avg_mt2_no_mentor(mt1):
    near = no_mentor.where("Midterm 1", are.between_or_equal_to(mt1-2, mt1+2))
    return near.column("Midterm 2").mean()

predicted_mt2 = scores.apply(avg_mt2_no_mentor, "Midterm 1")

In [ ]:
scores.drop(2).with_column('Predicted Mt2', predicted_mt2).scatter('Midterm 1')

In [ ]:
scores = scores.with_column("Improvement", scores.column('Midterm 2') - predicted_mt2)

scores.hist("Improvement", bins=np.arange(-30, 31, 5), group="Mentored", unit="point")

In [ ]:
def of_at_least_5(values):
    return sum(values >= 5) / len(values)

scores.select('Mentored', 'Improvement').group('Mentored', of_at_least_5).set_format(1, PercentFormatter)

In [ ]:
scores.group("Mentored", np.mean)

> **The flip:** raw averages made mentoring look bad (weaker students opt in — selection bias). Against a fair baseline, mentored students *improved* by several points. This is why Lecture 2's "association is not causation" was worth learning.

In [ ]:
def mean_ci(observations):
    means = []
    for i in np.arange(2000):
        means.append(observations.sample().column("Improvement").mean())
    lower, upper = percentile(2.5, means), percentile(97.5, means)
    print("Mean improvement:", observations.column("Improvement").mean())
    print("95% CI of mean improvement:", lower, "to", upper)

mentored = scores.where("Mentored", True)
mean_ci(mentored)

In [ ]:
mean_ci(mentored.where("Midterm 1", are.below(20)))

In [ ]:
mean_ci(mentored.where("Midterm 1", are.between(20, 30)))

In [ ]:
mean_ci(mentored.where("Midterm 1", are.above_or_equal_to(30)))

> **And the honest caveat:** the confidence intervals by score band are wide, and this still isn't a randomized experiment — we've *adjusted* for MT1, not for everything. The tools tell you what the data can and can't say. That's the course. 🎓

---
> **You made it.** Tables → visualization → simulation → inference → prediction → classification → decision-making: every tool in this notebook was built in this class, from `2 + 3` onward. Good luck on the final — you're ready.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### The prediction ladder
```python
scaled  = score / max1 * max2                   # naive
z_based = avg2 + z * sd2                        # assumes r = 1
regress = avg2 + r * z * sd2                    # the real one
```